# 06D — MSA Macro-Cluster Fallback Resolution

## Purpose

06C correctly fits each MSA macro K-Means model **only on complete Train MSA macro vectors**. Approximately 20% of loan rows cannot receive a direct MSA cluster because the MSA macro vector is structurally incomplete. 

06D **does not refit K-Means, does not impute MSA HPI, and never uses `Target_24M`**. Instead, it learns a Train-only fallback mapping from rows that have both (i) a valid MSA cluster and (ii) the corresponding State macro cluster.

The governing rule is:

\[
\boxed{\text{Train learns fallback} \rightarrow \text{Freeze} \rightarrow \text{Train/Validation apply}}
\]

For each of the four MSA specifications (Regular/LS × Profile A/B), the hierarchy is:

1. **Direct MSA assignment**: preserve the 06C MSA K-Means cluster exactly.
2. **State-cluster + calendar-year mapping**: for unresolved MSA rows, use the modal MSA cluster learned from complete **Train** rows sharing the same corresponding State cluster and year, but only when the cell has sufficient Train support and an acceptable modal share.
3. **State-cluster mapping**: if the year-specific cell is weak/unseen, back off to the modal MSA cluster within the same State macro cluster, learned from Train only.
4. **Global Train prior**: only if the State cluster itself is unresolved/unseen, use the Train modal MSA cluster. This guarantees a model-ready resolved column while making the weakest fallback explicit in the source audit.

The notebook preserves three distinct concepts:

- `*_Macro_Cluster_Raw`: direct 06C MSA K-Means assignment (`0/1/...` or `-1`).
- `*_Macro_Cluster_Resolved`: direct assignment where available, otherwise frozen Train-only fallback.
- `*_Macro_Cluster_Fallback_Applied`: `0` for direct MSA assignment, `1` for fallback-resolved assignment.

Additional audit-only/source columns record the fallback tier and confidence. **Validation is never used to learn a mapping.**

### Why this method

This approach avoids inventing missing MSA HPI values. It uses the already-available coarser State macro regime as a proxy for the missing fine-geography MSA regime, while keeping the provenance of fallback assignments observable. The year-conditioned tier allows the relationship between State and MSA regimes to change over time without fitting anything on Validation.


In [ ]:
# ======================================================================================
# 0. IMPORTS, PATHS, AND GLOBAL CONFIGURATION
# ======================================================================================
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# Notebook is expected to run from: project/notebooks/
ROOT = Path("..").resolve()
INPUT_PATH = ROOT / "data" / "output" / "final_modeling_dataset" / "SFLLD_feature_engineered_train_validation.parquet"
OUTPUT_PATH = ROOT / "data" / "output" / "final_modeling_dataset" / "SFLLD_feature_engineered_train_validation_MSA_fallback_resolved.parquet"
QA_DIR = ROOT / "data" / "output" / "quality_assurance" / "06D_MSA_macro_cluster_fallback_resolution"
ARTIFACT_DIR = QA_DIR / "artifacts"
QA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TRAIN = 160_003
EXPECTED_VALIDATION = 49_038
EXPECTED_TOTAL = 209_041

# Conservative support threshold for the most granular mapping tier.
# If a State-cluster × year cell has fewer Train complete-MSA rows than this,
# the algorithm backs off to State-cluster-only mapping.
MIN_CELL_N = 100

# A year-specific modal assignment below this purity is considered ambiguous
# and is backed off to State-cluster-only mapping.
MIN_CELL_MODAL_SHARE = 0.55

print("Input :", INPUT_PATH)
print("Output:", OUTPUT_PATH)
print("QA dir:", QA_DIR)
assert INPUT_PATH.exists(), f"Input parquet not found: {INPUT_PATH}"

Input : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation.parquet
Output: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation_MSA_fallback_resolved.parquet
QA dir: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06D_MSA_macro_cluster_fallback_resolution


In [ ]:
# ======================================================================================
# 1. LOAD 06C OUTPUT AND VERIFY DATASET IDENTITY
# ======================================================================================
df = pd.read_parquet(INPUT_PATH)

assert "Split" in df.columns
assert "Target_24M" in df.columns
assert "Loan_ID" in df.columns

split = df["Split"].astype("string").str.strip().str.lower()
train_mask = split.eq("train")
val_mask = split.eq("validation")

identity = pd.DataFrame({
    "Check": ["Train rows", "Validation rows", "Total rows", "Loan_ID unique"],
    "Observed": [int(train_mask.sum()), int(val_mask.sum()), len(df), bool(df["Loan_ID"].is_unique)],
    "Expected": [EXPECTED_TRAIN, EXPECTED_VALIDATION, EXPECTED_TOTAL, True],
})
identity["PASS"] = identity["Observed"].astype(str) == identity["Expected"].astype(str)
display(identity)
assert identity["PASS"].all(), "06C input identity check failed."

ORIGINAL_COLUMNS = df.columns.tolist()
ORIGINAL_LOAN_ID = df["Loan_ID"].copy()
ORIGINAL_TARGET = df["Target_24M"].copy()
ORIGINAL_SPLIT = df["Split"].copy()
print(f"Loaded {len(df):,} rows × {df.shape[1]:,} columns")

,Check,Observed,Expected,PASS
0,Train rows,160003,160003,True
1,Validation rows,49038,49038,True
2,Total rows,209041,209041,True
3,Loan_ID unique,True,True,True


Loaded 209,041 rows × 91 columns


In [ ]:
# ======================================================================================
# 2. RESOLVE THE ACTUAL 06C CLUSTER COLUMNS
# ======================================================================================
# Corrected 06C uses these names. 06D deliberately fails loudly if they are absent,
# rather than silently selecting a similarly named column.
SPEC = {
    "Regular_MSA_A": {
        "msa_raw_source": "Regular_MSA_A_Macro_Cluster_ResolvedBLS",
        "state_source": "Regular_State_A_Macro_Cluster_ResolvedBLS",
    },
    "Regular_MSA_B": {
        "msa_raw_source": "Regular_MSA_B_Macro_Cluster_ResolvedBLS",
        "state_source": "Regular_State_B_Macro_Cluster_ResolvedBLS",
    },
    "LS_MSA_A": {
        "msa_raw_source": "LS_MSA_A_Macro_Cluster_ResolvedBLS",
        "state_source": "LS_State_A_Macro_Cluster_ResolvedBLS",
    },
    "LS_MSA_B": {
        "msa_raw_source": "LS_MSA_B_Macro_Cluster_ResolvedBLS",
        "state_source": "LS_State_B_Macro_Cluster_ResolvedBLS",
    },
}

missing = []
for spec, cfg in SPEC.items():
    for role, col in cfg.items():
        if col not in df.columns:
            missing.append((spec, role, col))

if missing:
    print("Available macro-cluster columns:")
    print(*[c for c in df.columns if "Macro_Cluster" in c], sep="\n")
    raise KeyError(f"Required corrected-06C cluster columns are missing: {missing}")

# Time key: use First_Payment_Date because it is retained in the FE master and is
# observable at/near origination. It is used ONLY as a grouping key, never as target information.
assert "First_Payment_Date" in df.columns, "First_Payment_Date is required for the time-conditioned fallback."
fpd = pd.to_datetime(df["First_Payment_Date"], errors="coerce")
df["_06D_Fallback_Year"] = fpd.dt.year.astype("Int64")

print("Resolved source columns:")
for spec, cfg in SPEC.items():
    print(f"{spec:15s} | MSA={cfg['msa_raw_source']} | State={cfg['state_source']}")
print("Fallback year missing:", int(df["_06D_Fallback_Year"].isna().sum()))

Resolved source columns:
Regular_MSA_A   | MSA=Regular_MSA_A_Macro_Cluster_ResolvedBLS | State=Regular_State_A_Macro_Cluster_ResolvedBLS
Regular_MSA_B   | MSA=Regular_MSA_B_Macro_Cluster_ResolvedBLS | State=Regular_State_B_Macro_Cluster_ResolvedBLS
LS_MSA_A        | MSA=LS_MSA_A_Macro_Cluster_ResolvedBLS | State=LS_State_A_Macro_Cluster_ResolvedBLS
LS_MSA_B        | MSA=LS_MSA_B_Macro_Cluster_ResolvedBLS | State=LS_State_B_Macro_Cluster_ResolvedBLS
Fallback year missing: 0


In [4]:
# ======================================================================================
# 3. HELPER FUNCTIONS — TRAIN-ONLY FROZEN MAPPING
# ======================================================================================
def modal_summary(g, cluster_col):
    """Return deterministic modal cluster, modal count, N, and modal share."""
    vc = g[cluster_col].value_counts(dropna=False).sort_index()
    max_n = int(vc.max())
    # deterministic tie break: smallest cluster ID
    winners = sorted([int(k) for k, v in vc.items() if int(v) == max_n])
    mode = winners[0]
    n = int(vc.sum())
    return pd.Series({
        "Assigned_Cluster": mode,
        "Cell_N": n,
        "Modal_N": max_n,
        "Modal_Share": max_n / n if n else np.nan,
    })


def learn_fallback(train_df, raw_col, state_col, year_col):
    """
    Learn all mappings exclusively from Train rows with a direct MSA assignment.
    Target_24M is never referenced.
    """
    direct = train_df.loc[
        train_df[raw_col].notna()
        & train_df[raw_col].ge(0)
        & train_df[state_col].notna()
        & train_df[state_col].ge(0)
    ].copy()

    assert len(direct) > 0, f"No complete Train rows for {raw_col}."

    # Granular mapping: State macro cluster × calendar year -> MSA cluster
    sy = (
        direct.dropna(subset=[year_col])
        .groupby([state_col, year_col], observed=True, dropna=False)
        .apply(lambda g: modal_summary(g, raw_col), include_groups=False)
        .reset_index()
    )
    sy["Eligible"] = (
        sy["Cell_N"].ge(MIN_CELL_N)
        & sy["Modal_Share"].ge(MIN_CELL_MODAL_SHARE)
    )

    # Backoff: State macro cluster -> MSA cluster
    st = (
        direct.groupby(state_col, observed=True, dropna=False)
        .apply(lambda g: modal_summary(g, raw_col), include_groups=False)
        .reset_index()
    )

    # Final backoff: global Train mode
    vc = direct[raw_col].astype(int).value_counts().sort_index()
    max_n = int(vc.max())
    global_cluster = min(int(k) for k, v in vc.items() if int(v) == max_n)
    global_share = float(max_n / vc.sum())

    return {
        "state_year": sy,
        "state": st,
        "global_cluster": global_cluster,
        "global_share": global_share,
        "n_learning_rows": len(direct),
    }


def apply_frozen_fallback(frame, raw_col, state_col, year_col, learned):
    """Apply a frozen Train-only hierarchy without fitting anything."""
    raw = pd.to_numeric(frame[raw_col], errors="coerce")
    state = pd.to_numeric(frame[state_col], errors="coerce")
    year = frame[year_col]

    resolved = raw.copy()
    fallback = raw.isna() | raw.lt(0)
    source = pd.Series("DIRECT_MSA", index=frame.index, dtype="string")
    confidence = pd.Series(1.0, index=frame.index, dtype="float64")

    source.loc[fallback] = pd.NA
    confidence.loc[fallback] = np.nan

    sy = learned["state_year"].loc[lambda x: x["Eligible"]].copy()
    sy_lookup = {
        (int(r[state_col]), int(r[year_col])): (int(r["Assigned_Cluster"]), float(r["Modal_Share"]))
        for _, r in sy.dropna(subset=[state_col, year_col]).iterrows()
    }
    st = learned["state"].copy()
    st_lookup = {
        int(r[state_col]): (int(r["Assigned_Cluster"]), float(r["Modal_Share"]))
        for _, r in st.dropna(subset=[state_col]).iterrows()
    }

    # Tier 1: State cluster × year
    for idx in frame.index[fallback]:
        s = state.loc[idx]
        y = year.loc[idx]
        assigned = False
        if pd.notna(s) and s >= 0 and pd.notna(y):
            key = (int(s), int(y))
            if key in sy_lookup:
                c, p = sy_lookup[key]
                resolved.loc[idx] = c
                source.loc[idx] = "FALLBACK_STATE_CLUSTER_YEAR"
                confidence.loc[idx] = p
                assigned = True
        if assigned:
            continue

        # Tier 2: State cluster only
        if pd.notna(s) and s >= 0 and int(s) in st_lookup:
            c, p = st_lookup[int(s)]
            resolved.loc[idx] = c
            source.loc[idx] = "FALLBACK_STATE_CLUSTER"
            confidence.loc[idx] = p
            continue

        # Tier 3: global Train prior
        resolved.loc[idx] = int(learned["global_cluster"])
        source.loc[idx] = "FALLBACK_GLOBAL_TRAIN_PRIOR"
        confidence.loc[idx] = float(learned["global_share"])

    return (
        resolved.astype("int64"),
        fallback.astype("int8"),
        source,
        confidence,
    )


In [ ]:
# ======================================================================================
# 4. LEARN ON TRAIN, FREEZE, AND APPLY TO BOTH SPLITS
# ======================================================================================
train = df.loc[train_mask].copy()

mapping_store = {}
coverage_records = []
source_records = []

for spec, cfg in SPEC.items():
    raw_src = cfg["msa_raw_source"]
    state_src = cfg["state_source"]

    raw_out = f"{spec}_Macro_Cluster_Raw"
    resolved_out = f"{spec}_Macro_Cluster_Resolved"
    fallback_out = f"{spec}_Macro_Cluster_Fallback_Applied"
    source_out = f"{spec}_Macro_Cluster_Resolution_Source"
    conf_out = f"{spec}_Macro_Cluster_Fallback_Confidence"

    # Preserve the 06C assignment under an unambiguous Raw name.
    df[raw_out] = pd.to_numeric(df[raw_src], errors="coerce").fillna(-1).astype("int64")

    learned = learn_fallback(
        train_df=train,
        raw_col=raw_src,
        state_col=state_src,
        year_col="_06D_Fallback_Year",
    )
    mapping_store[spec] = learned

    resolved, fallback, source, confidence = apply_frozen_fallback(
        frame=df,
        raw_col=raw_src,
        state_col=state_src,
        year_col="_06D_Fallback_Year",
        learned=learned,
    )

    df[resolved_out] = resolved
    df[fallback_out] = fallback
    df[source_out] = source
    df[conf_out] = confidence

    for split_name, mask in [("Train", train_mask), ("Validation", val_mask)]:
        n = int(mask.sum())
        raw_valid = int((df.loc[mask, raw_out] >= 0).sum())
        unresolved_before = int((df.loc[mask, raw_out] < 0).sum())
        unresolved_after = int((df.loc[mask, resolved_out] < 0).sum())
        fallback_n = int(df.loc[mask, fallback_out].sum())
        coverage_records.append({
            "Specification": spec,
            "Split": split_name,
            "N": n,
            "Direct_MSA_N": raw_valid,
            "Direct_MSA_Pct": raw_valid / n * 100,
            "Unresolved_Before_N": unresolved_before,
            "Fallback_Applied_N": fallback_n,
            "Resolved_After_N": n - unresolved_after,
            "Resolved_After_Pct": (n - unresolved_after) / n * 100,
            "Unresolved_After_N": unresolved_after,
        })

        tmp = df.loc[mask, source_out].value_counts(dropna=False)
        for source_name, count in tmp.items():
            source_records.append({
                "Specification": spec,
                "Split": split_name,
                "Resolution_Source": str(source_name),
                "N": int(count),
                "Pct": int(count) / n * 100,
            })

coverage_table = pd.DataFrame(coverage_records)
source_table = pd.DataFrame(source_records)
print("Coverage before/after fallback:")
display(coverage_table)
print("Resolution source distribution:")
display(source_table)

Coverage before/after fallback:


,Specification,Split,N,Direct_MSA_N,Direct_MSA_Pct,Unresolved_Before_N,Fallback_Applied_N,Resolved_After_N,Resolved_After_Pct,Unresolved_After_N
0,Regular_MSA_A,Train,160003,128197,80.121623,31806,31806,160003,100.0,0
1,Regular_MSA_A,Validation,49038,40450,82.487051,8588,8588,49038,100.0,0
2,Regular_MSA_B,Train,160003,128197,80.121623,31806,31806,160003,100.0,0
3,Regular_MSA_B,Validation,49038,40450,82.487051,8588,8588,49038,100.0,0
4,LS_MSA_A,Train,160003,128197,80.121623,31806,31806,160003,100.0,0
5,LS_MSA_A,Validation,49038,40450,82.487051,8588,8588,49038,100.0,0
6,LS_MSA_B,Train,160003,128197,80.121623,31806,31806,160003,100.0,0
7,LS_MSA_B,Validation,49038,40450,82.487051,8588,8588,49038,100.0,0


Resolution source distribution:


,Specification,Split,Resolution_Source,N,Pct
0,Regular_MSA_A,Train,DIRECT_MSA,128197,80.121623
1,Regular_MSA_A,Train,FALLBACK_STATE_CLUSTER_YEAR,31720,19.824628
2,Regular_MSA_A,Train,FALLBACK_GLOBAL_TRAIN_PRIOR,86,0.053749
3,Regular_MSA_A,Validation,DIRECT_MSA,40450,82.487051
4,Regular_MSA_A,Validation,FALLBACK_STATE_CLUSTER_YEAR,6615,13.489539
5,Regular_MSA_A,Validation,FALLBACK_STATE_CLUSTER,1960,3.996900
6,Regular_MSA_A,Validation,FALLBACK_GLOBAL_TRAIN_PRIOR,13,0.026510
7,Regular_MSA_B,Train,DIRECT_MSA,128197,80.121623
8,Regular_MSA_B,Train,FALLBACK_STATE_CLUSTER_YEAR,31720,19.824628
9,Regular_MSA_B,Train,FALLBACK_GLOBAL_TRAIN_PRIOR,86,0.053749


In [6]:
# ======================================================================================
# 5. TRAIN-ONLY INTERNAL VALIDATION OF THE FALLBACK RULE
# ======================================================================================
# This diagnostic asks: among Train rows with a known direct MSA cluster, how accurately
# can the frozen proxy hierarchy reproduce that direct cluster when we pretend MSA is unavailable?
# It is NOT used to fit K-Means and does NOT use Target_24M.
validation_records = []

for spec, cfg in SPEC.items():
    raw = cfg["msa_raw_source"]
    state = cfg["state_source"]
    known = train[raw].notna() & train[raw].ge(0)
    eval_df = train.loc[known].copy()

    # Force the raw MSA cluster to unresolved in a temporary copy, then resolve using the
    # already-frozen mapping. This measures proxy agreement on the learning population.
    temp = eval_df.copy()
    temp["__raw_hidden__"] = -1
    pred, _, src, conf = apply_frozen_fallback(
        temp, "__raw_hidden__", state, "_06D_Fallback_Year", mapping_store[spec]
    )
    truth = pd.to_numeric(eval_df[raw], errors="coerce").astype(int)
    accuracy = float((pred.values == truth.values).mean())

    validation_records.append({
        "Specification": spec,
        "N_Known_Train_MSA": len(eval_df),
        "Proxy_Agreement_Accuracy": accuracy,
        "Mean_Fallback_Confidence": float(conf.mean()),
        "Min_Fallback_Confidence": float(conf.min()),
    })

proxy_validation = pd.DataFrame(validation_records)
print("Train-only proxy agreement diagnostic:")
display(proxy_validation)
print("\nNOTE: This is an interpretability/quality diagnostic, not a supervised target metric.")

Train-only proxy agreement diagnostic:


,Specification,N_Known_Train_MSA,Proxy_Agreement_Accuracy,Mean_Fallback_Confidence,Min_Fallback_Confidence
0,Regular_MSA_A,128197,0.740446,0.740446,0.551833
1,Regular_MSA_B,128197,0.890543,0.890543,0.765120
2,LS_MSA_A,128197,0.736967,0.742162,0.582220
3,LS_MSA_B,128197,0.892774,0.892774,0.817192



NOTE: This is an interpretability/quality diagnostic, not a supervised target metric.


In [7]:
# ======================================================================================
# 6. MANDATORY QA — LOGIC, LEAKAGE, PRESERVATION, AND RESOLUTION
# ======================================================================================
qa = []
def add_check(name, passed, detail=""):
    qa.append({"Check": name, "PASS": bool(passed), "Detail": str(detail)})

add_check("Row count unchanged", len(df) == EXPECTED_TOTAL, len(df))
add_check("Loan_ID unchanged", df["Loan_ID"].equals(ORIGINAL_LOAN_ID))
add_check("Target_24M unchanged", df["Target_24M"].equals(ORIGINAL_TARGET))
add_check("Split unchanged", df["Split"].equals(ORIGINAL_SPLIT))
add_check("No Target_24M used in mapping definition", True, "Mapping uses only MSA cluster, corresponding State cluster, and First_Payment_Date year.")

for spec, cfg in SPEC.items():
    raw_out = f"{spec}_Macro_Cluster_Raw"
    res_out = f"{spec}_Macro_Cluster_Resolved"
    fb_out = f"{spec}_Macro_Cluster_Fallback_Applied"
    src_out = f"{spec}_Macro_Cluster_Resolution_Source"

    direct = df[raw_out].ge(0)
    unresolved = ~direct

    add_check(f"{spec}: direct assignments unchanged",
              (df.loc[direct, res_out] == df.loc[direct, raw_out]).all())
    add_check(f"{spec}: fallback flag exactly equals raw unresolved mask",
              (df[fb_out].astype(bool) == unresolved).all())
    add_check(f"{spec}: no unresolved remains after fallback",
              df[res_out].ge(0).all(), int((df[res_out] < 0).sum()))
    add_check(f"{spec}: source complete",
              df[src_out].notna().all(), int(df[src_out].isna().sum()))
    add_check(f"{spec}: fallback count preserved",
              int(df[fb_out].sum()) == int(unresolved.sum()), int(df[fb_out].sum()))

qa_table = pd.DataFrame(qa)
print("Mandatory QA:")
display(qa_table)
assert qa_table["PASS"].all(), "06D mandatory QA failed. Do NOT save/freeze output."
print("ALL MANDATORY QA CHECKS PASSED.")

Mandatory QA:


,Check,PASS,Detail
0,Row count unchanged,True,209041
1,Loan_ID unchanged,True,
2,Target_24M unchanged,True,
3,Split unchanged,True,
4,No Target_24M used in mapping definition,True,"Mapping uses only MSA cluster, corresponding S..."
5,Regular_MSA_A: direct assignments unchanged,True,
6,Regular_MSA_A: fallback flag exactly equals ra...,True,
7,Regular_MSA_A: no unresolved remains after fal...,True,0
8,Regular_MSA_A: source complete,True,0
9,Regular_MSA_A: fallback count preserved,True,40394


ALL MANDATORY QA CHECKS PASSED.


In [8]:
# ======================================================================================
# 7. SAVE FROZEN MAPPING ARTIFACTS AND QA TABLES
# ======================================================================================
def serializable_mapping(spec, learned, state_col):
    sy = learned["state_year"].copy()
    st = learned["state"].copy()
    return {
        "specification": spec,
        "train_only": True,
        "time_key": "First_Payment_Date year",
        "state_cluster_column": state_col,
        "min_cell_n": MIN_CELL_N,
        "min_cell_modal_share": MIN_CELL_MODAL_SHARE,
        "global_train_cluster": int(learned["global_cluster"]),
        "global_train_modal_share": float(learned["global_share"]),
        "n_learning_rows": int(learned["n_learning_rows"]),
        "state_year_mapping": sy.to_dict(orient="records"),
        "state_mapping": st.to_dict(orient="records"),
    }

for spec, cfg in SPEC.items():
    artifact = serializable_mapping(spec, mapping_store[spec], cfg["state_source"])
    with open(ARTIFACT_DIR / f"06D_{spec}_fallback_mapping.json", "w", encoding="utf-8") as f:
        json.dump(artifact, f, indent=2, default=str)

coverage_table.to_csv(QA_DIR / "06D_MSA_fallback_coverage.csv", index=False)
source_table.to_csv(QA_DIR / "06D_MSA_resolution_source_distribution.csv", index=False)
proxy_validation.to_csv(QA_DIR / "06D_train_only_fallback_proxy_agreement.csv", index=False)
qa_table.to_csv(QA_DIR / "06D_quality_assurance_checks.csv", index=False)

print("Saved frozen mapping artifacts and QA tables.")

Saved frozen mapping artifacts and QA tables.


In [9]:
# ======================================================================================
# 8. SAVE FINAL 06D DATASET
# ======================================================================================
# Remove the temporary helper year. Resolution-source/confidence columns are retained
# intentionally for provenance and downstream sensitivity analysis.
df_out = df.drop(columns=["_06D_Fallback_Year"]).copy()

df_out.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df_out.shape[0]:,} × {df_out.shape[1]:,}")

Saved: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation_MSA_fallback_resolved.parquet
Shape: 209,041 × 111


In [10]:
# ======================================================================================
# 9. POST-SAVE RELOAD VALIDATION — REQUIRED BEFORE 06D CAN PASS
# ======================================================================================
re = pd.read_parquet(OUTPUT_PATH)
post = []
def post_check(name, passed, detail=""):
    post.append({"Check": name, "PASS": bool(passed), "Detail": str(detail)})

post_check("Reload row count", len(re) == len(df_out), len(re))
post_check("Reload column count", re.shape[1] == df_out.shape[1], re.shape[1])
post_check("Reload Loan_ID order", re["Loan_ID"].equals(df_out["Loan_ID"]))
post_check("Reload Target_24M exact", re["Target_24M"].equals(df_out["Target_24M"]))
post_check("Reload Split exact", re["Split"].equals(df_out["Split"]))

for spec in SPEC:
    raw = f"{spec}_Macro_Cluster_Raw"
    res = f"{spec}_Macro_Cluster_Resolved"
    fb = f"{spec}_Macro_Cluster_Fallback_Applied"
    src = f"{spec}_Macro_Cluster_Resolution_Source"
    conf = f"{spec}_Macro_Cluster_Fallback_Confidence"

    for c in [raw, res, fb, src, conf]:
        post_check(f"{spec}: {c} persisted", c in re.columns)
    post_check(f"{spec}: resolved exact after reload", re[res].equals(df_out[res]))
    post_check(f"{spec}: fallback flag exact after reload", re[fb].equals(df_out[fb]))
    post_check(f"{spec}: 100% resolved after reload", re[res].ge(0).all(), int((re[res] < 0).sum()))

post_table = pd.DataFrame(post)
display(post_table)
post_table.to_csv(QA_DIR / "06D_post_save_validation.csv", index=False)
assert post_table["PASS"].all(), "06D post-save validation failed."

summary = {
    "input_file": str(INPUT_PATH),
    "output_file": str(OUTPUT_PATH),
    "rows": int(len(re)),
    "columns": int(re.shape[1]),
    "train_rows": int(train_mask.sum()),
    "validation_rows": int(val_mask.sum()),
    "validation_used_for_learning": False,
    "target_used_for_learning": False,
    "fallback_method": "Train-only hierarchical modal mapping: State macro cluster + First_Payment_Date year -> State macro cluster -> global Train prior",
    "min_cell_n": MIN_CELL_N,
    "min_cell_modal_share": MIN_CELL_MODAL_SHARE,
    "mandatory_qa_passed": bool(qa_table["PASS"].all()),
    "post_save_qa_passed": bool(post_table["PASS"].all()),
}
with open(QA_DIR / "06D_final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=" * 120)
print("06D COMPLETE — MSA MACRO-CLUSTER FALLBACK RESOLUTION PASSED")
print("=" * 120)
print(json.dumps(summary, indent=2))

,Check,PASS,Detail
0,Reload row count,True,209041
1,Reload column count,True,111
2,Reload Loan_ID order,True,
3,Reload Target_24M exact,True,
4,Reload Split exact,True,
5,Regular_MSA_A: Regular_MSA_A_Macro_Cluster_Raw...,True,
6,Regular_MSA_A: Regular_MSA_A_Macro_Cluster_Res...,True,
7,Regular_MSA_A: Regular_MSA_A_Macro_Cluster_Fal...,True,
8,Regular_MSA_A: Regular_MSA_A_Macro_Cluster_Res...,True,
9,Regular_MSA_A: Regular_MSA_A_Macro_Cluster_Fal...,True,


06D COMPLETE — MSA MACRO-CLUSTER FALLBACK RESOLUTION PASSED
{
  "input_file": "C:\\Users\\itwill\\Desktop\\study\\\ucf54\ub529\uacf5\ubd80\\\ud504\ub85c\uc81d\ud2b8\\The Single Family Loan-Level Dataset (SFLLD) - 6\\data\\output\\final_modeling_dataset\\SFLLD_feature_engineered_train_validation.parquet",
  "output_file": "C:\\Users\\itwill\\Desktop\\study\\\ucf54\ub529\uacf5\ubd80\\\ud504\ub85c\uc81d\ud2b8\\The Single Family Loan-Level Dataset (SFLLD) - 6\\data\\output\\final_modeling_dataset\\SFLLD_feature_engineered_train_validation_MSA_fallback_resolved.parquet",
  "rows": 209041,
  "columns": 111,
  "train_rows": 160003,
  "validation_rows": 49038,
  "validation_used_for_learning": false,
  "target_used_for_learning": false,
  "fallback_method": "Train-only hierarchical modal mapping: State macro cluster + First_Payment_Date year -> State macro cluster -> global Train prior",
  "min_cell_n": 100,
  "min_cell_modal_share": 0.55,
  "mandatory_qa_passed": true,
  "post_save_qa_passed"

## Interpretation of the 06D output

A resolved cluster is **not** equivalent to a directly observed/complete MSA macro vector. Downstream modeling should therefore retain the fallback indicator alongside the resolved cluster whenever the resolved cluster is used.

Recommended modeling pair for each specification:

\[
\boxed{\text{MSA Macro Cluster Resolved} + \text{Fallback Applied Indicator}}
\]

This lets a model distinguish a direct fine-geography regime assignment from a proxy assignment. The raw cluster remains available for sensitivity analysis and for reproducing the original 06C coverage.

### Required review before freezing 06D

The notebook must finish with all mandatory and post-save QA checks equal to `True`. In addition, inspect:

- the proportion assigned by each fallback tier;
- `Proxy_Agreement_Accuracy` for each specification;
- fallback confidence distributions;
- Train vs Validation resolved cluster shares.

A 100% resolved rate alone is **not** evidence that the fallback is informative. The proxy-agreement and source-distribution diagnostics are what determine whether the fallback is substantively useful enough for the final model matrix.
